# Matched-training robustness study at 130M parameters

Runs `scripts/scale_study.py` on one GPU (A100 recommended, L4 works).

Every arm starts from the same public checkpoint (`state-spaces/mamba-130m-hf`) and gets the same data, optimiser, steps and seeds:
unconstrained vs. constrained, clean vs. adversarial fine-tuning, plus single-constraint ablations.
Every attack is re-optimised against each fine-tuned checkpoint.

Constrained arms come in two versions. `lipcal` uses constants calibrated to the checkpoint, so the model starts as almost the same function. `lip` uses the paper's constants, which change the pretrained function drastically.

**Runtime → Change runtime type → A100 GPU (or L4).** Run the cells in order. Results are written to Google Drive after every stage, so a disconnected session resumes where it stopped: just run the same cells again.

No Kaggle or Hugging Face token is needed: the checkpoint, tokenizer and WikiText-2 are public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type"
gpu = torch.cuda.get_device_name(0); PRESET = "a100" if "A100" in gpu else "l4"
print("GPU:", gpu, "| preset:", PRESET)

## 1. Output folder on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/lipmamba_scale_study"
import os; os.makedirs(OUT, exist_ok=True); os.environ["OUT"] = OUT; os.environ["PRESET"] = PRESET

## 2. Code

In [ ]:
%cd /content
!rm -rf CV && git clone -q --depth 1 -b claude/implement-lipmamba-model-SHbBY https://github.com/rogerpanel/CV.git
%cd /content/CV/lipmamba
!pip -q install transformers datasets pytest
os.environ["PYTHONPATH"] = "/content/CV/lipmamba/src"
!git log --oneline -1

## 3. Checks (about 2 minutes)
The first command confirms that the converted model computes the same function as the `transformers` Mamba implementation. The second runs the whole pipeline on a toy model.

In [ ]:
!python -m pytest -q tests/test_pretrained.py
!python scripts/scale_study.py run --model lip_at --tiny --out /content/smoke/lip_at_s0.json

## 4. Pilot: timing and a first look

100 fine-tuning steps and a very small attack run, for the unconstrained and the calibrated constrained model. Check in the output of **both** runs:

* `init`: perplexity of `lipcal` should be close to that of `mamba` (about 26), not astronomically larger;
* the `step` lines: `acc` should reach about 1.0, and `s/step` gives the training speed;
* the `L=8` line: its time in minutes, for 16 examples and 5 search iterations.

Paste the output (or upload `pilot/*.json`) before starting section 5 if anything looks off.

In [ ]:
!python scripts/scale_study.py run --model mamba  --seed 0 --preset $PRESET --steps 100 --lengths 8 --n-eval 16 --iters 5 --force --out $OUT/pilot/mamba_s0.json
!python scripts/scale_study.py run --model lipcal --seed 0 --preset $PRESET --steps 100 --lengths 8 --n-eval 16 --iters 5 --force --out $OUT/pilot/lipcal_s0.json

## 5. Main study, seed 0 (five arms)

`mamba_released`, `mamba`, `lipcal`, `mamba_at`, `lipcal_at`, 400 fine-tuning steps each. Finished arms are skipped when you re-run the cell. Each arm prints its own timing.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 -- --preset $PRESET

## 6. Paper constants, ablations and more seeds
Run these when section 5 has finished, in this order, as far as your compute allows.

1. `lip`, `lip_at`: the paper's constants, same budget.
2. Four single-constraint ablations (seed 0), which answer the request for an ablation with attack success rate.
3. Seeds 1 and 2 of everything, for error bars.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 --paper-constants -- --preset $PRESET
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 --paper-constants --ablations -- --preset $PRESET
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 1 2 --paper-constants --ablations -- --preset $PRESET

## 7. Summary and download
Upload `lipmamba_scale_results.zip` (a few hundred kilobytes: JSON files only) to the chat.

In [ ]:
!python scripts/scale_study.py aggregate --dir $OUT/main
!cd $OUT && zip -qr /content/lipmamba_scale_results.zip main pilot -x '*.pt'
from google.colab import files; files.download('/content/lipmamba_scale_results.zip')